In [1]:
import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'

In [2]:
%load_ext autoreload
%autoreload 2

import sys; sys.path.append('..')
import MeshFEM, mesh, elastic_sheet, elastic_solid, energy, tensors, benchmark
import meshing, triangulation, py_newton_optimizer
from io_redirection import suppress_stdout as so
import sheet_convergence, sim_utils, semisphere_convergence
from tri_mesh_viewer import TriMeshViewer
import numpy as np, time
import copy

In [3]:
thickness= 0.01
mV = 2e-5
myres = 7

In [4]:
tm = semisphere_convergence.getCreasedHemisphereTetMesh(thickness,maxVol=mV)
esolid = semisphere_convergence.getElasticSolid(tm)

In [5]:
opts = py_newton_optimizer.NewtonOptimizerOptions()
opts.niter = 200
opts.gradTol = 1e-12

In [6]:
benchmark.reset()
esolid_sim,t1 = semisphere_convergence.gravitySimulation(esolid, opts=opts)
benchmark.report()
esolid_energy = esolid_sim.energy()
print("Solid Mesh Energy: ", esolid_sim.energy())

0	2.7274e-07	1.39857e-09	1	0
1	2.72692e-07	1.98541e-08	1	0
2	2.72692e-07	2.13233e-11	1	0
3	2.72692e-07	7.81742e-15	1	0
Compress Matrix	1.13473	1
Newton iterations	21.4923	1
    Newton iterate	21.4846	4
        Backtracking	0.0192349	3
            energy	0.0127916	3
        Compute descent direction	21.3321	3
            newton_step	21.3321	3
                Catamari Symbolic Factorize	7.51893	1
                    CatamariConverter	0.657998	1
                    Construct plan	0.117606	1
                        Build	0.0851679	1
                    SparseLDL.Factor	0.86654	1
                        supernodal_ldl.Factorization.Factor	0.866535	1
                            FormSupernodes	0.865129	1
                                FormSupernodes	0.601154	1
                                InitializeFactors	0.263924	1
                    cholmod_l_nested_dissection	5.87248	1
                CholeskyFactorizerBase.solve	0.402966	3
                    Catamari Solve	0.389324	3
              

In [7]:
opts.factorizer = opts.factorizer.CatamariNesdis

opt.factorizer = opts.factorizer.Catamari

In [8]:
m, mcreases = semisphere_convergence.getCreasedHemisphereSheetMesh(thickness,resolution=myres)
esheet = semisphere_convergence.getElasticSheet(m, thickness, creases=mcreases)

In [9]:
benchmark.reset()
esheet_sim, t2 = semisphere_convergence.gravitySimulation(esheet, opts=opts)
print(esheet_sim.energy(), esheet_sim.energy(etype=esheet_sim.EnergyType.Membrane), esheet_sim.energy(etype=esheet_sim.EnergyType.Bending))
benchmark.report()
esheet_energy = esheet_sim.energy()

0	1.6473e-20	1.70466e-22	0	0
0	2.8984e-07	2.12004e-09	1	0
1	2.89791e-07	7.55544e-10	1	0
2	2.89791e-07	6.24927e-13	1	0
5.0216109417471856e-11 3.2958564821212285e-11 1.7257544596259562e-11
Newton iterations	3.17586	2
    Newton iterate	3.16873	4
        Backtracking	0.0499959	2
        Compute descent direction	3.04296	2
            newton_step	3.04296	2
                Catamari Symbolic Factorize	2.40644	1
                    CatamariConverter	0.099448	1
                    Construct plan	0.0249021	1
                        Build	0.0185149	1
                    SparseLDL.Factor	0.212691	1
                        supernodal_ldl.Factorization.Factor	0.212687	1
                            FormSupernodes	0.212098	1
                                FormSupernodes	0.138123	1
                                InitializeFactors	0.073971	1
                    cholmod_l_nested_dissection	2.06889	1
                CholeskyFactorizerBase.solve	0.040488	2
                    Catamari Solve	0.0372229	2


In [10]:
energy_rel_error = np.abs(esolid_energy-esheet_energy)/esolid_energy
print("Relative Error of Energy: ",energy_rel_error*100, "%")

Relative Error of Energy:  3.402515856674105 %


In [11]:
# Visualization

In [12]:
from ipywidgets import HBox
vdefo, sheetView, sampledSolidView = semisphere_convergence.visVertexStrainOnSheetAndSolid(esolid_sim, esheet_sim)
HBox([sheetView.show(), sampledSolidView.show(), vdefo.show()])

In [13]:
vertex_strain_rel_error,_,_,_,_,_ = semisphere_convergence.computeVertexStrain(esolid_sim, esheet_sim)
print("Relative Error of Vertex Strain: ", vertex_strain_rel_error*100, "%")

Relative Error of Vertex Strain:  162.73202257612826 %
